# Proposed validation — review before it runs

**What this measures:** `dice_gap` (segformer3d_dice − segresnet_dice) measures whether the real `SegFormer3D` module, trained from scratch under an identical fixed-budget protocol as `SegResNet`, reaches segmentation-accuracy parity with an established MONAI net on the real, cached Task01_BrainTumour 8/4 subset fetched once via `monai.apps.DecathlonDataset`.

**Target metric:** `dice_gap`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

The executable copy lives at `eval/eval_segformer3d_brats_parity.py`, which is what `.remyx/validation.yaml` points at; keep the two in step, or point `suite:` here if you would rather maintain the notebook.

In [1]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

In [2]:
# Parameters
variant = "feature"
ref = "3e8ab7c35c913d937e6bd765530f16537a1040d1"
seed = 0


## Execution context

The cells below are the script at `eval/eval_segformer3d_brats_parity.py`, unchanged. This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [3]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "eval/eval_segformer3d_brats_parity.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

[remyx] cwd /workspace/target_repo | script /workspace/target_repo/eval/eval_segformer3d_brats_parity.py


In [4]:
#!/usr/bin/env python
"""Fixed-budget, from-scratch Dice-parity check: SegFormer3D vs SegResNet on the real,
cached Task01_BrainTumour 8/4 subset (Decathlon dataset, fetched once via
monai.apps.DecathlonDataset). Runs against both the pre-change (`dev`) checkout and the
PR head: SegFormer3D is unimportable on `dev`, so that arm's import is wrapped in a
narrow try/except and degrades to zeroed metrics on baseline, while the PR head trains
and scores both models for real, per validation.yaml.
"""

"Fixed-budget, from-scratch Dice-parity check: SegFormer3D vs SegResNet on the real,\ncached Task01_BrainTumour 8/4 subset (Decathlon dataset, fetched once via\nmonai.apps.DecathlonDataset). Runs against both the pre-change (`dev`) checkout and the\nPR head: SegFormer3D is unimportable on `dev`, so that arm's import is wrapped in a\nnarrow try/except and degrades to zeroed metrics on baseline, while the PR head trains\nand scores both models for real, per validation.yaml.\n"

In [5]:
from __future__ import annotations

import argparse
import json
import os
import random
import statistics
import sys
import time

In [6]:
import torch

REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, REPO_ROOT)

SEED = 0

In [7]:
def dice_coefficient(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> float:
    pred = pred.reshape(pred.shape[0], -1)
    target = target.reshape(target.shape[0], -1)
    inter = (pred * target).sum(-1)
    denom = pred.sum(-1) + target.sum(-1)
    return ((2 * inter + eps) / (denom + eps)).mean().item()

In [8]:
def build_dataset(root_dir: str, n_total: int, crop):
    from monai.apps import DecathlonDataset
    from monai.transforms import CenterSpatialCropd, Compose, EnsureChannelFirstd, Lambdad, LoadImaged

    transform = Compose(
        [
            LoadImaged(keys=["image", "label"]),
            EnsureChannelFirstd(keys=["image", "label"]),
            CenterSpatialCropd(keys=["image", "label"], roi_size=crop),
            Lambdad(keys="label", func=lambda x: (x > 0).float()),
        ]
    )
    ds = DecathlonDataset(
        root_dir=root_dir,
        task="Task01_BrainTumour",
        section="training",
        transform=transform,
        download=True,
        seed=SEED,
        val_frac=0.0,
        cache_num=0,
        num_workers=0,
        progress=False,
    )
    indices = random.Random(SEED).sample(range(len(ds)), n_total)
    items = [ds[i] for i in indices]
    return items

In [9]:
def train_and_eval(model, train_images, train_labels, val_images, val_labels, device, n_iters):
    from monai.losses import DiceLoss

    model.to(device)
    train_images, train_labels = train_images.to(device), train_labels.to(device)
    val_images, val_labels = val_images.to(device), val_labels.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = DiceLoss(sigmoid=True, squared_pred=True)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    warmup_n = min(5, max(0, n_iters - 1))
    model.train()
    times = []
    for it in range(n_iters):
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        opt.zero_grad()
        out = model(train_images)
        loss = loss_fn(out, train_labels)
        loss.backward()
        opt.step()
        if device.type == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        if it >= warmup_n:
            times.append(t1 - t0)

    median_iter_time = statistics.median(times) if times else 0.0
    train_time_s = median_iter_time * n_iters
    max_mem_mb = torch.cuda.max_memory_allocated(device) / 1e6 if device.type == "cuda" else 0.0

    model.eval()
    with torch.no_grad():
        val_out = model(val_images)
        pred = (torch.sigmoid(val_out) > 0.5).float()
        dice = dice_coefficient(pred, val_labels)

    n_params = sum(p.numel() for p in model.parameters())
    return dice, n_params, train_time_s, max_mem_mb

In [10]:
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--variant", default=None)
    parser.add_argument("--ref", default=None)
    parser.add_argument("--seed", default=None)
    parser.parse_known_args()

    from monai.utils import set_determinism

    set_determinism(seed=SEED)
    random.seed(SEED)

    # --- guardrail: existing nets in the shared nets/__init__.py still import cleanly ---
    existing_net_names = ["SegResNet", "UNETR", "SwinUNETR", "VNet", "BasicUNet", "AttentionUnet", "DynUNet"]
    ok = 0
    for name in existing_net_names:
        try:
            module = __import__("monai.networks.nets", fromlist=[name])
            getattr(module, name)
            ok += 1
        except (ImportError, AttributeError):
            pass
    existing_nets_import_success_rate = ok / len(existing_net_names)

    # --- defensive import of the symbol this diff adds: fails cleanly on the `dev` baseline ---
    try:
        from monai.networks.nets import SegFormer3D

        has_segformer3d = True
    except (ImportError, AttributeError):
        SegFormer3D = None
        has_segformer3d = False

    from monai.networks.nets import SegResNet

    smoke = os.environ.get("REMYX_SMOKE") == "1"
    crop = (32, 32, 32) if smoke else (64, 64, 64)
    n_total = 4 if smoke else 12
    n_train = 2 if smoke else 8
    n_iters = 2 if smoke else 200

    root_dir = os.path.join(REPO_ROOT, ".cache", "brats_data")
    os.makedirs(root_dir, exist_ok=True)

    items = build_dataset(root_dir, n_total, crop)
    train_items, val_items = items[:n_train], items[n_train:]
    train_images = torch.stack([it["image"] for it in train_items])
    train_labels = torch.stack([it["label"] for it in train_items])
    val_images = torch.stack([it["image"] for it in val_items])
    val_labels = torch.stack([it["label"] for it in val_items])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    in_channels = train_images.shape[1]

    # --- SegResNet arm (unaffected by this diff, runs identically on both baseline and head) ---
    set_determinism(seed=SEED)
    segresnet = SegResNet(spatial_dims=3, in_channels=in_channels, out_channels=1, init_filters=8)
    segresnet_dice, segresnet_params, segresnet_train_time_s, segresnet_max_mem_mb = train_and_eval(
        segresnet, train_images, train_labels, val_images, val_labels, device, n_iters
    )

    # --- SegFormer3D arm: degrades to zeros on the `dev` baseline where it can't be imported ---
    if has_segformer3d:
        set_determinism(seed=SEED)
        segformer3d = SegFormer3D(in_channels=in_channels, out_channels=1)
        segformer3d_dice, segformer3d_params, segformer3d_train_time_s, segformer3d_max_mem_mb = train_and_eval(
            segformer3d, train_images, train_labels, val_images, val_labels, device, n_iters
        )
    else:
        segformer3d_dice = 0.0
        segformer3d_params = 0
        segformer3d_train_time_s = 0.0
        segformer3d_max_mem_mb = 0.0

    dice_gap = segformer3d_dice - segresnet_dice

    metrics = {
        "dice_gap": dice_gap,
        "existing_nets_import_success_rate": existing_nets_import_success_rate,
        "segformer3d_dice": segformer3d_dice,
        "segresnet_dice": segresnet_dice,
        "segformer3d_params": segformer3d_params,
        "segresnet_params": segresnet_params,
        "segformer3d_train_time_s": segformer3d_train_time_s,
        "segresnet_train_time_s": segresnet_train_time_s,
        "segformer3d_max_mem_mb": segformer3d_max_mem_mb,
        "segresnet_max_mem_mb": segresnet_max_mem_mb,
    }
    print(json.dumps(metrics))

if __name__ == "__main__":
    main()

/root/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-15 00:39:08,595 - INFO - Verified 'Task01_BrainTumour.tar', md5: 240a19d752f0d9e9101544901065d872.


2026-09-15 00:39:08,596 - INFO - Downloaded: /workspace/target_repo/.cache/brats_data/Task01_BrainTumour.tar


2026-09-15 00:39:08,597 - INFO - Writing into directory: /workspace/target_repo/.cache/brats_data.


{"dice_gap": 0.12091624736785889, "existing_nets_import_success_rate": 1.0, "segformer3d_dice": 0.7291803359985352, "segresnet_dice": 0.6082640886306763, "segformer3d_params": 4251585, "segresnet_params": 1176825, "segformer3d_train_time_s": 18.13344879997203, "segresnet_train_time_s": 33.07347380000465, "segformer3d_max_mem_mb": 805.983232, "segresnet_max_mem_mb": 1403.575808}


## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
loop: {max_iterations: 8, fix_code: true}
benchmarks:
  - name: segformer3d-brats-architecture-parity
    suite: "eval/eval_segformer3d_brats_parity.py"
    packages: [nibabel]
    scorer: dice_gap
    baseline: dev
    metrics:
      - name: dice_gap
        role: target
        direction: max
        threshold: -0.05
      - name: existing_nets_import_success_rate
        role: guardrail
        direction: max
        threshold: 1.0
      - name: segformer3d_dice
        role: cost
        direction: max
        threshold: 0.0
      - name: segresnet_dice
        role: cost
        direction: max
        threshold: 0.0
      - name: segformer3d_params
        role: cost
        direction: min
        threshold: 10000000
      - name: segresnet_params
        role: cost
        direction: min
        threshold: 10000000
      - name: segformer3d_train_time_s
        role: cost
        direction: min
        threshold: 1800
      - name: segresnet_train_time_s
        role: cost
        direction: min
        threshold: 1800
      - name: segformer3d_max_mem_mb
        role: cost
        direction: min
        threshold: 40000
      - name: segresnet_max_mem_mb
        role: cost
        direction: min
        threshold: 40000
    policy: {guardrail_veto: true}
held_constant:
  - "same fixed 8/4 subset of the real Task01_BrainTumour training list: deterministically selected with one fixed seed for both models, identical inputs for both arms"
  - "same crop geometry: (64, 64, 64) voxels, same seed, same crop/resize pipeline applied to the real Task01_BrainTumour volumes for both models"
  - "same optimizer, learning rate and loss: Adam(lr=1e-4) with DiceLoss(sigmoid=True, squared_pred=True), identical for both models"
  - "same fixed step budget: 200 from-scratch training iterations for both models, no pretrained weights for either"
  - "same torch/numpy/monai seed across both models via monai.utils.set_determinism, single seed only, no multi-seed averaging"
  - "Task01_BrainTumour is fetched exactly once via monai.apps.DecathlonDataset(root_dir=<working dir>, task='Task01_BrainTumour', download=True) and cached in the working directory; repeat runs reuse the cache rather than re-downloading"
avoid:
  - "this is a fixed-budget, from-scratch comparison on a fixed real 8/4 Task01_BrainTumour subset -- a parity signal, not a reproduction of either paper's full training protocol or fully converged accuracy on all of BraTS"
  - "the multi-gigabyte Task01_BrainTumour download is now exercised per user_guidance; it is fetched once into the working directory and cached, so repeat invocations of this script must detect the existing cache and skip re-downloading to stay practical"
  - "baseline: dev is retained: the script's SegFormer3D import is wrapped in a narrow try/except (ImportError, AttributeError) so it fails cleanly on the dev checkout, where has_segformer3d is False and that arm's metrics degrade to zero, while SegResNet still trains for real on both arms -- this is the required baseline-vs-head execution, not a two-arm delta on the target itself"
  - "train time, parameter counts and peak device memory are reported only as cost references, never as a pass/fail gate on model quality"
  - "no invented paper Dice numbers: only Dice computed by this script's own from-scratch training run on the real cached subset is reported"
  - "single seed only, as instructed: no seed-to-seed variance estimate exists for dice_gap, so the fixed -0.05 tolerance band remains a fixed bound rather than a noise-derived one"
compute:
  tier: gpu
  timeout_s: 5400
provenance:
  dice_gap: "user_guidance (fixed-budget from-scratch Dice comparison on the real Task01_BrainTumour 8/4 subset, computed within the feature/head run; the dev checkout produces a degraded, zeroed value for this metric via the script's defensive import)"
  existing_nets_import_success_rate: "inferred -- regression guardrail for the monai/networks/nets/__init__.py edit this PR makes, since that file is shared by every existing net; measured on both dev and head"
  segformer3d_dice: "user_guidance (mean validation Dice per model; zero on dev, where SegFormer3D cannot be imported)"
  segresnet_dice: "user_guidance (mean validation Dice per model)"
  segformer3d_params: "user_guidance (params reported as a cost reference only)"
  segresnet_params: "user_guidance (params reported as a cost reference only)"
  segformer3d_train_time_s: "user_guidance (train time reported as a cost reference only; median-per-iteration extrapolation after a warm-up, per gpu-tier measurement practice)"
  segresnet_train_time_s: "user_guidance (train time reported as a cost reference only; median-per-iteration extrapolation after a warm-up, per gpu-tier measurement practice)"
  segformer3d_max_mem_mb: "inferred -- peak device memory via torch.cuda.max_memory_allocated, required for gpu-tier wall-clock/device measurements"
  segresnet_max_mem_mb: "inferred -- peak device memory via torch.cuda.max_memory_allocated, required for gpu-tier wall-clock/device measurements"
  baseline: "corrected back to `dev`: the script's defensive import of SegFormer3D already fails cleanly and degrades to zeroed metrics on the dev checkout per system rule 2, so there is no need to skip running the script against baseline -- the target metric's value is still read from the head run against a fixed bound, not a cross-arm delta"
  packages: "inferred -- nibabel added because monai.apps.DecathlonDataset/LoadImaged reads real NIfTI volumes and nibabel is not part of MONAI's core pyproject dependencies"
  held_constant: "user_guidance (same fixed 8/4 subset and seed, downloaded once via monai.apps.DecathlonDataset) refined by protocol_doc:monai/networks/nets/segformer3d.py for the channel/crop geometry"
  compute: "inferred -- timeout_s raised from 3600 to 5400 to cover the one-time Task01_BrainTumour download alongside the unchanged 200-iteration x2-model training budget"
  suite: "synthesized (R1 maturity repo: tests + CI only, no BraTS benchmark harness exists to run (a) against); loads the real cached Task01_BrainTumour subset via monai.apps.DecathlonDataset per user_guidance instead of synthetic volumes; execution bugs from the prior attempt (nondeterministic subset selection, a re-download-on-every-run caching bug) are fixed in this revision per user_guidance 'fix the issues in the test and run again'; the baseline field is restored to `dev` since the earlier switch to `none` was itself a bug, not a fix, per system rule 2"
```